# Stacked BiLSTM · validation v2

Whitespace BPE → Embedding 128 → 3-layer BiLSTM, hidden 256 per direction → Dropout → Linear 512 → 3.

Train on v2 train, select best.pt on VAL, and export predictions with logits for VAL, holdout and TEST. Checkpoints are backed up to Drive after every epoch.


## Setup



In [1]:
import os
import sys
import base64
import subprocess
from pathlib import Path

ENVIRONMENT = "colab" if "google.colab" in sys.modules else "kaggle"
root = Path("/content" if ENVIRONMENT == "colab" else "/kaggle/working")
repo_dir = Path(os.environ.get("UA_NER_REPO_DIR", root / "UA-Locations-NER-Deep-Learning"))

if not repo_dir.exists():
    if ENVIRONMENT == "colab":
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    else:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env = os.environ.copy()
    env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.extraheader",
               GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {auth}")
    try:
        subprocess.run(["git", "clone", "-b", "stacked",
                        "https://github.com/dkalchenko01/UA-Locations-NER-Deep-Learning.git",
                        str(repo_dir)], env=env, check=True)
    finally:
        del token, auth, env

sys.path.insert(0, str(repo_dir))
print(repo_dir)

/content/UA-Locations-NER-Deep-Learning


In [2]:
%pip install -q -r "$repo_dir/requirements.txt"

In [3]:
import importlib
for module in ("src.data_io", "src.metrics", "src.training", "src.bigru_helpers", "src.experiment_tracking"):
    importlib.reload(importlib.import_module(module))

import gc
import shutil
import json
import platform
import uuid
from importlib.metadata import version

import pandas as pd
import torch
from tokenizers import Tokenizer

from src.data_io import (get_data_dir, load_manifest, load_data_splits,
                         save_json, save_predictions, file_sha256)
from src.bigru_helpers import CLEAN_OPTIONS, backup_files, postprocessed_scores
from src.dataset import prepare_ner_records, prepare_inference_records, make_loader, LABEL_TO_ID
from src.model import BiLSTMTagger
from src.metrics import competition_scores
from src.postprocessing import postprocess
from src.training import set_seed, train_epoch, predict, save_checkpoint

import wandb
from time import perf_counter
from src.experiment_tracking import start_run, log_epoch, finish_run

## W&B login


In [4]:
os.environ["WANDB_BASE_URL"] = "https://api.forge.coreweave.com"
os.environ["WANDB_DISABLE_CODE"] = "true"
os.environ["WANDB_CONSOLE"] = "off"

if ENVIRONMENT == "colab":
    from google.colab import userdata
    wandb.login(key=userdata.get("WANDB_API_KEY"), relogin=True)
else:
    from kaggle_secrets import UserSecretsClient
    wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"), relogin=True)

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.forge.coreweave.com.
wandb: No netrc file found, creating one.
wandb: Appending key for api.forge.coreweave.com to your netrc file: /root/.netrc
wandb: Currently logged in as: daldoshyna (daldoshyna-kyiv-school-of-economics) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


## Configuration



In [5]:
config = {
    "model": "bilstm", "architecture": "stacked", "embedding_dim": 128, "hidden_size": 256,
    "num_layers": 3, "dropout": 0.2, "seed": 17, "batch_size": 128,
    "learning_rate": 1e-3, "epochs": 5, "gradient_clip": 1.0,
    "max_padded_tokens": 16384, "prepare_batch_size": 2000,
    "sources": ["uk", "ru"], "optimizer": "Adam",
    "tokenizer": "bpe_whitespace.json", "selection_metric": "uk_comp_f1",
    "label_to_id": LABEL_TO_ID, "max_sequence_length": None,
    "early_stopping": False, "environment": ENVIRONMENT,
    "validation_version": "v2", "manifest_name": "validation_manifest_v2", "metric_version": "kaggle_original_regex_v1",
    "postprocessing_version": "expand_trim_empty",
    "decoding": "viterbi_lines", "clean_options": dict(CLEAN_OPTIONS), "upload_checkpoint": False,
}


## Run files and provenance


In [6]:
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
set_seed(config["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
run_id = "stacked_" + pd.Timestamp.now(tz="UTC").strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
run_dir = root / "outputs" / run_id
run_dir.mkdir(parents=True, exist_ok=False)
splits_dir = repo_dir / "data/splits"
tokenizer_path = repo_dir / "artifacts/tokenizers" / config["tokenizer"]
manifest_path = splits_dir / f"{config['manifest_name']}.parquet"
from google.colab import drive
drive.mount("/content/drive")
backup_dir = Path("/content/drive/MyDrive/UA-Locations-NER/stacked") / run_id
backup_dir.mkdir(parents=True, exist_ok=True)

provenance = {
    "run_id": run_id, "seed": config["seed"], "python": platform.python_version(),
    "git_revision": subprocess.run(["git", "-C", str(repo_dir), "rev-parse", "HEAD"],
                                   capture_output=True, text=True).stdout.strip() or None,
    "git_status": subprocess.run(["git", "-C", str(repo_dir), "status", "--short"],
                                 capture_output=True, text=True).stdout.strip(),
    "packages": {name: version(name) for name in ["torch", "tokenizers", "numpy", "pandas", "pyarrow", "tqdm", "wandb"]},
    "device": str(device), "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0) if device.type == "cuda" else None,
    "tokenizer_sha256": file_sha256(tokenizer_path),
    "manifest_sha256": file_sha256(manifest_path),
    "source_sha256": {path.name: file_sha256(path) for path in sorted((repo_dir / "src").glob("*.py"))},
    "notebook_sha256": file_sha256(repo_dir / "notebooks/04_bilstm_stacked.ipynb"),
}
save_json(run_dir / "config.json", config)
save_json(run_dir / "provenance.json", provenance)
backup_files(backup_dir, run_dir / "config.json", run_dir / "provenance.json", tokenizer_path)
print(device, run_dir, "\nDrive:", backup_dir)

Mounted at /content/drive
cuda /content/outputs/stacked_20261118T171209Z_0eafba6e 
Drive: /content/drive/MyDrive/UA-Locations-NER/stacked/stacked_20261118T171209Z_0eafba6e


## Experiment tracking


In [7]:
os.environ["WANDB_DIR"] = str(run_dir)
run = start_run(
    model="bilstm", config=config, name=run_id,
    validation_version=config["validation_version"], metric_version=config["metric_version"],
    postprocessing_version=config["postprocessing_version"], repo_dir=repo_dir,
    split_manifest_path=manifest_path,
    tokenizer_path=tokenizer_path, provenance=provenance,
    tags=["stacked", f"layers-{config['num_layers']}", f"hidden-{config['hidden_size']}"]
)
save_json(run_dir / "wandb_run.json", {"id": run.id, "url": run.url,
                                      "path": f"{run.entity}/{run.project}/{run.id}"})
backup_files(backup_dir, run_dir / "wandb_run.json")
print(run.url)

https://app.forge.coreweave.com/daldoshyna-kyiv-school-of-economics/ua-location-ner/runs/t6bdccnv


## Data



In [8]:
tokenizer = Tokenizer.from_file(str(tokenizer_path))
tokenizer.no_padding()
tokenizer.no_truncation()
pad_id = tokenizer.token_to_id("[PAD]")
data_dir = get_data_dir(ENVIRONMENT)
manifest = load_manifest(splits_dir, name=config["manifest_name"])
display(pd.crosstab(manifest["source"], manifest["split"]))

record_sets = {"train": [], "val": [], "holdout": []}
alignment = []
for source in config["sources"]:
    splits = load_data_splits(ENVIRONMENT, splits_dir, sources=(source,),
                              manifest_name=config["manifest_name"])
    for split, target in record_sets.items():
        frame = splits.pop(f"{source}_{split}")
        records, skipped = prepare_ner_records(
            tokenizer, frame, source, split, batch_size=config["prepare_batch_size"])
        target.extend(records)
        alignment.append({"source": source, "split": split, "total": len(frame),
                          "retained": len(records), "alignment_failures": len(skipped)})
        save_json(run_dir / f"{source}_{split}_skipped.json", skipped)
        del frame, records
    del splits
    gc.collect()
train_records, val_records, holdout_records = (
    record_sets["train"], record_sets["val"], record_sets["holdout"])
for split in ("val", "holdout"):
    expected = manifest[(manifest["split"] == split) & manifest["source"].isin(config["sources"])]
    assert {r["text_id"] for r in record_sets[split]} == {
        f"{row.source}_{row.row_id}" for row in expected.itertuples()}
alignment = pd.DataFrame(alignment)
display(alignment)
alignment.to_csv(run_dir / "alignment.csv", index=False)
backup_files(backup_dir, run_dir / "alignment.csv")

100%|██████████| 811M/811M [00:23<00:00, 35.5MB/s]

Extracting files...


split,holdout,train,val
source,,,
ru,24743,100000,24626
uk,100757,805983,100741


Prepare uk/train:   0%|          | 0/403 [00:00<?, ?it/s]

Prepare uk/val:   0%|          | 0/51 [00:00<?, ?it/s]

Prepare uk/holdout:   0%|          | 0/51 [00:00<?, ?it/s]

Prepare ru/train:   0%|          | 0/50 [00:00<?, ?it/s]

Prepare ru/val:   0%|          | 0/13 [00:00<?, ?it/s]

Prepare ru/holdout:   0%|          | 0/13 [00:00<?, ?it/s]

,source,split,total,retained,alignment_failures
0,uk,train,805983,804367,1616
1,uk,val,100741,100741,169
2,uk,holdout,100757,100757,206
3,ru,train,100000,99909,91
4,ru,val,24626,24626,17
5,ru,holdout,24743,24743,21


In [9]:
loader_options = {
    "pad_id": pad_id, "batch_size": config["batch_size"], "seed": config["seed"],
    "max_padded_tokens": config["max_padded_tokens"], "pin_memory": device.type == "cuda"
}
train_loader = make_loader(train_records, shuffle=True, **loader_options)
val_loader = make_loader(val_records, **loader_options)

holdout_loader = make_loader(holdout_records, **loader_options)

print("Train:", len(train_records), "VAL:", len(val_records), "Holdout:", len(holdout_records))

Train: 904276 VAL: 125367 Holdout: 125500


## Model



In [10]:
model_config = {
    "vocab_size": tokenizer.get_vocab_size(), "pad_id": pad_id,
    "embedding_dim": config["embedding_dim"], "hidden_size": config["hidden_size"],
    "num_layers": config["num_layers"], "dropout": config["dropout"], "num_labels": len(LABEL_TO_ID)
}
set_seed(config["seed"])
model = BiLSTMTagger(**model_config).to(device)
optimizer_class = getattr(torch.optim, config["optimizer"])
optimizer = optimizer_class(model.parameters(), lr=config["learning_rate"])
save_json(run_dir / "model_config.json", model_config)
backup_files(backup_dir, run_dir / "model_config.json")
print(model)
print("Parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

run.config.update({"model_config": model_config})

BiLSTMTagger(
  (embedding): Embedding(32000, 128, padding_idx=0)
  (lstm): LSTM(128, 256, num_layers=3, batch_first=True, dropout=0.2, bidirectional=True)
  (dropout): Dropout(p=0.2, inplace=False)
  (classifier): Linear(in_features=512, out_features=3, bias=True)
)
Parameters: 8041987


## Training



In [11]:
history, best_score, best_epoch = [], -1, None
best_path = run_dir / "best.pt"
for epoch in range(1, config["epochs"] + 1):
    epoch_started = perf_counter()
    train_loader.batch_sampler.set_epoch(epoch)
    train_loss = train_epoch(model, train_loader, optimizer, device, config["gradient_clip"])
    predictions, val_losses = predict(model, val_loader, device, decoding=config["decoding"])
    scores = postprocessed_scores(val_records, predictions, config["clean_options"])
    raw_scores = competition_scores(val_records, predictions)
    result = {"epoch": epoch, "train_loss": train_loss,
              "epoch_seconds": perf_counter() - epoch_started,
              **{f"{source}_val_loss": value for source, value in val_losses.items()},
              **scores, **{f"raw_{key}": value for key, value in raw_scores.items()}}
    history.append(result)
    pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)
    epoch_path = run_dir / f"epoch_{epoch:02d}.pt"
    save_checkpoint(epoch_path, model, optimizer, epoch, config, model_config, scores, provenance)
    if scores[config["selection_metric"]] > best_score:
        best_score, best_epoch = scores[config["selection_metric"]], epoch
        save_checkpoint(best_path, model, optimizer, epoch, config, model_config, scores, provenance)
    backup_files(backup_dir, epoch_path, best_path, run_dir / "history.csv")
    log_epoch(run, epoch, result, selection_metric=config["selection_metric"],
              epoch_seconds=result["epoch_seconds"],
              extra_metrics={f"val_{config['validation_version']}/raw/{key}": value
                             for key, value in raw_scores.items()})
    print(result)
    del predictions
display(pd.DataFrame(history))
optimizer.zero_grad(set_to_none=True)
del optimizer
gc.collect()

Train:   0%|          | 0/7102 [00:01<?, ?it/s]

Predict:   0%|          | 0/985 [00:00<?, ?it/s]

{'epoch': 1, 'train_loss': 0.03413134095150467, 'epoch_seconds': 304.65736362199993, 'uk_val_loss': 0.018628775404699545, 'ru_val_loss': 0.02519446553143481, 'combined_val_loss': 0.020262367723404544, 'uk_comp_f1': 0.8649994, 'ru_comp_f1': 0.8839847, 'combined_comp_f1': 0.86980622, 'precision': 0.8822679341567503, 'recall': 0.8576916322767583, 'TP': 39073, 'FP': 5214, 'FN': 6483, 'raw_uk_comp_f1': 0.86850878, 'raw_ru_comp_f1': 0.85487687, 'raw_combined_comp_f1': 0.86506416, 'raw_precision': 0.8727463640222515, 'raw_recall': 0.8575160242339099, 'raw_TP': 39065, 'raw_FP': 5696, 'raw_FN': 6491}


Train:   0%|          | 0/7104 [00:01<?, ?it/s]

Predict:   0%|          | 0/985 [00:00<?, ?it/s]

{'epoch': 2, 'train_loss': 0.01630093855236554, 'epoch_seconds': 310.28496406599993, 'uk_val_loss': 0.01661913053157171, 'ru_val_loss': 0.020678422862163676, 'combined_val_loss': 0.017629112520156736, 'uk_comp_f1': 0.87687132, 'ru_comp_f1': 0.91111111, 'combined_comp_f1': 0.88555545, 'precision': 0.8847308341184461, 'recall': 0.8863815962771094, 'TP': 40380, 'FP': 5261, 'FN': 5176, 'raw_uk_comp_f1': 0.88266768, 'raw_ru_comp_f1': 0.8968254, 'raw_combined_comp_f1': 0.88625147, 'raw_precision': 0.8816692735488356, 'raw_recall': 0.8908815523750988, 'raw_TP': 40585, 'raw_FP': 5447, 'raw_FN': 4971}


... (7 outputs left out)

Predict:   0%|          | 0/985 [00:00<?, ?it/s]

{'epoch': 5, 'train_loss': 0.007122493367359534, 'epoch_seconds': 307.7619623459998, 'uk_val_loss': 0.018633420754908096, 'ru_val_loss': 0.02330660703037359, 'combined_val_loss': 0.01979614410251056, 'uk_comp_f1': 0.88473196, 'ru_comp_f1': 0.916326, 'combined_comp_f1': 0.89277725, 'precision': 0.9008984846452998, 'recall': 0.8848011238914743, 'TP': 40308, 'FP': 4434, 'FN': 5248, 'raw_uk_comp_f1': 0.89155059, 'raw_ru_comp_f1': 0.90609126, 'raw_combined_comp_f1': 0.89524482, 'raw_precision': 0.8997405937520786, 'raw_recall': 0.8907937483536746, 'raw_TP': 40581, 'raw_FP': 4522, 'raw_FN': 4975}


,epoch,train_loss,epoch_seconds,uk_val_loss,ru_val_loss,combined_val_loss,uk_comp_f1,ru_comp_f1,combined_comp_f1,precision,...,FP,FN,raw_uk_comp_f1,raw_ru_comp_f1,raw_combined_comp_f1,raw_precision,raw_recall,raw_TP,raw_FP,raw_FN
0,1,0.034131,304.657364,0.018629,0.025194,0.020262,0.864999,0.883985,0.869806,0.882268,...,5214,6483,0.868509,0.854877,0.865064,0.872746,0.857516,39065,5696,6491
1,2,0.016301,310.284964,0.016619,0.020678,0.017629,0.876871,0.911111,0.885555,0.884731,...,5261,5176,0.882668,0.896825,0.886251,0.881669,0.890882,40585,5447,4971
2,3,0.011998,309.090192,0.016228,0.020444,0.017277,0.882020,0.916736,0.890782,0.894399,...,4772,5139,0.889195,0.898384,0.891509,0.891040,0.891979,40635,4969,4921
3,4,0.009065,303.752651,0.017030,0.021351,0.018105,0.881743,0.915877,0.890405,0.887252,...,5173,4848,0.889035,0.905540,0.893214,0.886475,0.900057,41003,5251,4553
4,5,0.007122,307.761962,0.018633,0.023307,0.019796,0.884732,0.916326,0.892777,0.900898,...,4434,5248,0.891551,0.906091,0.895245,0.899741,0.890794,40581,4522,4975


1024

## Select checkpoint


In [14]:
checkpoint_path = run_dir / "epoch_03.pt"
checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
selected_epoch = checkpoint["epoch"]
summary = {"selected_epoch": selected_epoch, "checkpoint": checkpoint_path.name,
           "selection_metric": config["selection_metric"], "scores": checkpoint["scores"],
           "decoding": config["decoding"], "clean_options": config["clean_options"]}
save_json(run_dir / "summary.json", summary)
backup_files(backup_dir, run_dir / "summary.json")
print(summary)
del checkpoint

{'selected_epoch': 3, 'checkpoint': 'epoch_03.pt', 'selection_metric': 'uk_comp_f1', 'scores': {'uk_comp_f1': 0.88201999, 'ru_comp_f1': 0.9167358, 'combined_comp_f1': 0.89078186, 'precision': 0.8943990794219833, 'recall': 0.8871937834752832, 'TP': 40417, 'FP': 4772, 'FN': 5139}, 'decoding': 'viterbi_lines', 'clean_options': {'expand': True, 'trim': True, 'empty': True}}


## Export VAL, holdout and TEST with logits

The selected best.pt is loaded above. All three exports use the same weights and tokenizer. Spans and logits are saved before cleanup. The package is saved directly to /content/drive/MyDrive/ner_results/<model>/<run>/, including checkpoint.pt, tokenizer.json and metadata.json.


In [15]:
export_dir = Path("/content/drive/MyDrive/ner_results") / config["model"] / (
    f"{run_id}_{checkpoint_path.stem}_{config['decoding']}")
export_dir.mkdir(parents=True, exist_ok=False)
test = pd.read_csv(data_dir / "test.csv", usecols=["text_id", "text"])
test_records = prepare_inference_records(tokenizer, test)
test_loader = make_loader(test_records, **loader_options)
for name, records, loader in [
    ("val", val_records, val_loader),
    ("holdout", holdout_records, holdout_loader),
    ("test", test_records, test_loader),
]:
    predictions, _ = predict(model, loader, device, decoding=config["decoding"], return_logits=True)
    save_predictions(export_dir / f"{name}.jsonl", records, predictions)
    if name == "test":
        test_predictions = predictions
    del predictions
    gc.collect()
shutil.copy2(checkpoint_path, export_dir / "checkpoint.pt")
shutil.copy2(tokenizer_path, export_dir / "tokenizer.json")
save_json(export_dir / "metadata.json", {
    "checkpoint": "checkpoint.pt", "checkpoint_original_name": checkpoint_path.name,
    "checkpoint_sha256": file_sha256(checkpoint_path), "epoch": selected_epoch,
    "model_config": model_config, "training_config": config, "provenance": provenance,
    "tokenizer_sha256": file_sha256(tokenizer_path),
    "manifest_sha256": file_sha256(manifest_path), "label_to_id": LABEL_TO_ID,
    "decoding": config["decoding"], "postprocessing": "none",
    "val_rows": len(val_records), "holdout_rows": len(holdout_records),
    "test_rows": len(test_records),
})
print("Saved on Drive:", export_dir)

Predict:   0%|          | 0/985 [00:00<?, ?it/s]

Predict:   0%|          | 0/987 [00:00<?, ?it/s]

Predict:   0%|          | 0/6 [00:00<?, ?it/s]

Saved on Drive: /content/drive/MyDrive/ner_results/bilstm/stacked_20261118T171209Z_0eafba6e_epoch_03_viterbi_lines


## Submission

Uses the TEST predictions from the export cell above with the same cleanup used for checkpoint selection.


In [16]:
submission = test[["text_id"]].copy()
submission["locations"] = [
    json.dumps(postprocess(row.text, test_predictions[row.text_id]["spans"],
                           **config["clean_options"]), ensure_ascii=False)
    for row in test.itertuples(index=False)
]
path = run_dir / "submission_viterbi_lines_clean_v5.csv"
submission.to_csv(path, index=False)
backup_files(backup_dir, path)
print(path)
display(submission.head())

/content/outputs/stacked_20261118T171209Z_0eafba6e/submission_viterbi_lines_clean_v5.csv


,text_id,locations
0,0,"[""Київ""]"
1,1,"[""Києві"", ""Києва"", ""Шулявського шляхопроводу""]"
2,2,"[""Гоголеве"", ""Миргородського району"", ""Полтавс..."
3,3,[]
4,4,"[""проспекті Академіка Глушкова, 9""]"


## Finish tracking


In [17]:
finish_run(run, best_epoch=selected_epoch, best_score=best_score,
           selection_metric=config["selection_metric"], checkpoint_path=checkpoint_path,
           upload_checkpoint=config["upload_checkpoint"])

epoch,▁▃▅▆█
time/epoch_seconds,▂█▇▁▅
train/loss,█▃▂▂▁
val_v2/FN,█▂▂▁▃
val_v2/FP,██▄▇▁
val_v2/TP,▁▇▇█▆
val_v2/combined_comp_f1,▁▆▇▇█
val_v2/combined_loss,█▂▁▃▇
val_v2/precision,▁▂▆▃█
val_v2/raw/FN,█▃▂▁▃
+12,...
